In [1]:
import base64
import json
import os
from io import BytesIO

import pandas as pd
import requests
from datasets import concatenate_datasets, load_dataset
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

load_dotenv()

d:\youtube\TheAIGuy\NLP\doubleword\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

In [3]:
dataset = concatenate_datasets(
    [dataset["train"], dataset["validation"], dataset["test"]]
)
dataset

Dataset({
    features: ['image', 'ground_truth'],
    num_rows: 1000
})

In [4]:
dataset = dataset.add_column("index", list(range(1, len(dataset) + 1)))
dataset

Dataset({
    features: ['image', 'ground_truth', 'index'],
    num_rows: 1000
})

In [5]:
dataset[0]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=864x1296>,
 'ground_truth': '{"gt_parse": {"menu": [{"nm": "Nasi Campur Bali", "cnt": "1 x", "price": "75,000"}, {"nm": "Bbk Bengil Nasi", "cnt": "1 x", "price": "125,000"}, {"nm": "MilkShake Starwb", "cnt": "1 x", "price": "37,000"}, {"nm": "Ice Lemon Tea", "cnt": "1 x", "price": "24,000"}, {"nm": "Nasi Ayam Dewata", "cnt": "1 x", "price": "70,000"}, {"nm": "Free Ice Tea", "cnt": "3 x", "price": "0"}, {"nm": "Organic Green Sa", "cnt": "1 x", "price": "65,000"}, {"nm": "Ice Tea", "cnt": "1 x", "price": "18,000"}, {"nm": "Ice Orange", "cnt": "1 x", "price": "29,000"}, {"nm": "Ayam Suir Bali", "cnt": "1 x", "price": "85,000"}, {"nm": "Tahu Goreng", "cnt": "2 x", "price": "36,000"}, {"nm": "Tempe Goreng", "cnt": "2 x", "price": "36,000"}, {"nm": "Tahu Telor Asin", "cnt": "1 x", "price": "40,000."}, {"nm": "Nasi Goreng Samb", "cnt": "1 x", "price": "70,000"}, {"nm": "Bbk Panggang Sam", "cnt": "3 x", "price": "366,000"}, {"nm": 

In [6]:
schema_dict = {
    "menu": {
        "nm": "name of the menu",
        "num": "identification number of menu",
        "unitprice": "unit price of menu",
        "cnt": "quantity of menu",
        "discountprice": "discounted price of menu",
        "price": "total price of menu",
        "itemsubtotal": "price of each menu after discount applied",
        "vatyn": "whether the price includes tax or not",
        "etc": "others",
        "sub": {
            "nm": "name of submenu",
            "unitprice": "unit price of submenu",
            "cnt": "quantity of submenu",
            "price": "total price of submenu",
            "etc": "others",
        },
    },
    "sub_total": {
        "price": "subtotal price",
        "discount_price": "discounted price in total",
        "service_price": "service charge",
        "othersvc_price": "added charge other than service charge",
        "tax_price": "tax amount",
        "etc": "others",
    },
    "total": {
        "total_price": "total price",
        "etc": "others",
        "cashprice": "amount of price paid in cash",
        "changeprice": "amount of change in cash",
        "creditcardprice": "amount of price paid in credit/debit card",
        "emoneyprice": "amount of price paid in emoney, point",
        "menutype_cnt": "total count of type of menu",
        "menuqty_cnt": "total count of quantity",
    },
}

In [7]:
example1 = """{
  'menu': [
    {'nm': 'Item A', 'cnt': '1 x', 'price': '71,000'},
    {'nm': 'Item B', 'cnt': '1 x', 'price': '128,000'},
    {'nm': 'Item C', 'cnt': '1 x', 'price': '39,000'},
    {'nm': 'Item D', 'cnt': '1 x', 'price': '22,000'},
    {'nm': 'Item E', 'cnt': '1 x', 'price': '74,000'},
    {'nm': 'Item F', 'cnt': '3 x', 'price': '0'},
    {'nm': 'Item G', 'cnt': '1 x', 'price': '63,000'},
    {'nm': 'Item H', 'cnt': '1 x', 'price': '17,000'},
    {'nm': 'Item I', 'cnt': '1 x', 'price': '31,000'},
    {'nm': 'Item J', 'cnt': '1 x', 'price': '88,000'},
    {'nm': 'Item K', 'cnt': '2 x', 'price': '34,000'},
    {'nm': 'Item L', 'cnt': '2 x', 'price': '35,000'},
    {'nm': 'Item M', 'cnt': '1 x', 'price': '42,000.'},
    {'nm': 'Item N', 'cnt': '1 x', 'price': '72,000'},
    {'nm': 'Item O', 'cnt': '3 x', 'price': '359,000'},
    {'nm': 'Item P', 'cnt': '1 x', 'price': '95,000'},
    {'nm': 'Item Q', 'cnt': '2 x', 'price': '46,000'},
    {'nm': 'Item R', 'cnt': '1 x', 'price': '30,000'},
    {'nm': 'Item S', 'cnt': '1 x', 'price': '41,000'},
    {'nm': 'Item T', 'cnt': '1 x', 'price': '0'},
    {'nm': 'Item U', 'cnt': '1 x', 'price': '47,000'},
    {'nm': 'Item V', 'cnt': '1 x', 'price': '19,000'}
  ],
  'sub_total': {
    'subtotal_price': '1,298,000',
    'service_price': '102,300',
    'tax_price': '139,800',
    'etc': '-50'
  },
  'total': {
    'total_price': '1,540,050'
  }
}
"""

In [8]:
example2 = """{
  'menu': [
    {'nm': 'Item A', 'cnt': '1', 'price': '61,000'},
    {'nm': 'Item B',
     'cnt': '1',
     'price': '168,000',
     'sub': {'nm': 'Option X'}},
    {'nm': 'Item C',
     'cnt': '1',
     'price': '198,000',
     'sub': {'nm': 'Option Y'}},
    {'nm': 'Item D', 'cnt': '1', 'price': '24,000'},
    {'nm': 'Item E', 'cnt': '1', 'price': '30,000'},
    {'nm': 'Item F', 'cnt': '1', 'price': '38,000'}
  ],
  'sub_total': {
    'subtotal_price': '519,000',
    'service_price': '26,400',
    'tax_price': '54,600'
  },
  'total': {
    'total_price': '600,000'
  }
}
"""

In [9]:
example3 = """{
  'menu': [
    {'nm': 'Item A', 'cnt': '4', 'price': '95,000'},
    {'nm': 'Item B', 'cnt': '4', 'price': '82,000'},
    {'nm': 'Item C', 'cnt': '3', 'price': '63,000'},
    {'nm': 'Item D', 'cnt': '2', 'price': '45,000'},
    {'nm': 'Item E', 'cnt': '3', 'price': '59,000'}
  ],
  'sub_total': {
    'subtotal_price': '344,000'
  },
  'total': {
    'total_price': '344,000',
    'cashprice': '360,000',
    'changeprice': '-16,000',
    'menutype_cnt': '5',
    'menuqty_cnt': '16'
  }
}
"""

In [10]:
system_prompt = f"""You are a Vision Language Model designed to extract structured data from invoice receipts.
    Task:
    Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

    Requirements:
    1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
    2. Preserve exact formatting for all the extracted values.  
    3. Do not output fields that lack data—omit empty keys.  
    4. Do not add any information not present in the invoice.
    5. In case of prices and currencies, ensure to maintain the original format without any modifications.

    Schema:
    {schema_dict}

    Few-shot Examples:

    Example 1:
    {example1}

    ---

    Example 2:
    {example2}

    ---
    Example 3:
    {example3}

    Output:
    Return valid, minimal JSON matching this schema - no extraneous keys or null values.
    """

In [11]:
def pil_to_base64(pil_image):
    buffer = BytesIO()
    pil_image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [12]:
def generate_line(idx, image_base64):
    request = {
        "custom_id": f"request-{idx}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "Qwen/Qwen3-VL-235B-A22B-Instruct-FP8",
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt,
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{image_base64}"
                            },
                        },
                    ],
                },
            ],
        },
    }
    return request

In [13]:
with open("invoice_extraction_cord_1000_samples.jsonl", "w") as f:
    # iterate through each row of `dataset`
    for row in tqdm(dataset):
        idx = row["index"]
        image = row["image"]
        gt = row["ground_truth"]

        image_base64 = pil_to_base64(image)

        line = generate_line(idx, image_base64)
        f.write(json.dumps(line) + "\n")

100%|██████████| 1000/1000 [00:49<00:00, 20.16it/s]


In [14]:
client = OpenAI(
    api_key=os.getenv("DOUBLEWORDKEY"), base_url="https://api.doubleword.ai/v1"
)

with open("invoice_extraction_cord_1000_samples.jsonl", "rb") as file:
    batch_file = client.files.create(file=file, purpose="batch")

print(f"File ID: {batch_file.id}")


File ID: 58542f54-22e1-4457-b18f-1b0626121a3a


In [15]:
batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
)

print(f"Batch ID: {batch.id}")

Batch ID: ad7c5933-2fca-41cc-94fb-bd582e5c85e3


In [23]:
batch_status = client.batches.retrieve("dd100cfc-37d0-4ad0-8fdc-44066497f470")
print(f"Status: {batch_status.status}")

Status: completed


In [24]:
batch_status.output_file_id

'0c781350-99f1-4655-b43a-fe89f8cde5a6'

In [25]:
# Get file output ID
batch_output_file_id = batch_status.output_file_id

# Download file content
url = f"https://api.doubleword.ai/v1/files/{batch_output_file_id}/content"
headers = {"Authorization": f"Bearer {os.getenv('DOUBLEWORDKEY')}"}

response = requests.get(url, headers=headers)

# Check if file is incomplete (batch still running)
is_incomplete = response.headers.get("X-Incomplete") == "true"
last_line = response.headers.get("X-Last-Line")

# Save to file
with open("batch-output.jsonl", "wb") as f:
    f.write(response.content)

if is_incomplete:
    print(f"Partial file downloaded (up to line {last_line})")
    print(f"To resume from this point: add ?offset={last_line} to the URL")
else:
    print("Complete file downloaded!")

Complete file downloaded!


### Load the results

In [27]:
result_lines = []
with open("batch-output.jsonl", "r") as f:
    for line in f:
        result = json.loads(line)
        idx = result["custom_id"]
        response = result["response"]["body"]["choices"][0]["message"]["content"]
        try:
            response = json.loads(response)
        except json.JSONDecodeError:
            continue
        idx = idx.replace("request-", "")

        result_lines.append((idx, response))

result_df = pd.DataFrame(result_lines, columns=["index", "response"])
result_df["index"] = result_df["index"].astype(int)

result_df

,index,response
0,11,"{'menu': [{'nm': 'Viet Milk Coffee', 'cnt': '1..."
1,74,"{'menu': [{'nm': '2005-CHEESE JOHN', 'cnt': 'x..."
2,62,"{'menu': [{'nm': 'TRIPPLE CHEESE', 'cnt': '1',..."
3,33,"{'menu': [{'nm': 'Ketoprak', 'cnt': '1', 'pric..."
4,87,"{'menu': [{'nm': 'BUNCIS MUDA', 'cnt': '1', 'p..."
...,...,...
994,708,"{'menu': [{'nm': 'CHEESE ROLLS 6PCS', 'cnt': '..."
995,624,"{'menu': [{'nm': 'IKAN GURAME REG', 'cnt': '1'..."
996,675,"{'menu': [{'nm': 'UDANG REBUS (M)', 'cnt': '1'..."
997,960,"{'menu': [{'nm': 'Tahu Isi', 'cnt': '2', 'pric..."


In [29]:
dataset_df = dataset.select_columns(["index", "ground_truth"]).to_pandas()

# parse ground_truth (if string) and extract the 'gt_parse' dict
dataset_df["ground_truth"] = dataset_df["ground_truth"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)
dataset_df["gt_parse"] = dataset_df["ground_truth"].apply(
    lambda d: d.get("gt_parse") if isinstance(d, dict) else None
)

# verify
dataset_df = dataset_df[["index", "gt_parse"]]
dataset_df

,index,gt_parse
0,1,"{'menu': [{'nm': 'Nasi Campur Bali', 'cnt': '1..."
1,2,"{'menu': [{'nm': 'SPGTHY BOLOGNASE', 'cnt': '1..."
2,3,"{'menu': [{'nm': 'HAKAU UDANG', 'cnt': '4', 'p..."
3,4,"{'menu': [{'nm': 'Bintang Bremer', 'cnt': '1',..."
4,5,"{'menu': {'nm': 'BASO BIHUN', 'unitprice': '43..."
...,...,...
995,996,"{'menu': [{'nm': 'BASO TAHU', 'unitprice': '43..."
996,997,"{'menu': {'nm': 'BBQ Chicken', 'cnt': '1', 'pr..."
997,998,"{'menu': [{'nm': 'BUBUR GO', 'unitprice': '20...."
998,999,"{'menu': [{'nm': 'BLACK PAPPER MEATBALL PAS', ..."


In [30]:
final_result_df = dataset_df.merge(result_df, on="index", how="inner")
final_result_df

,index,gt_parse,response
0,1,"{'menu': [{'nm': 'Nasi Campur Bali', 'cnt': '1...","{'menu': [{'nm': 'Nasi Campur Bali', 'cnt': '1..."
1,2,"{'menu': [{'nm': 'SPGTHY BOLOGNASE', 'cnt': '1...","{'menu': [{'nm': 'SPGTHY BOLOGNASE', 'cnt': '1..."
2,3,"{'menu': [{'nm': 'HAKAU UDANG', 'cnt': '4', 'p...","{'menu': [{'nm': 'HAKAU UDANG', 'cnt': '4', 'p..."
3,4,"{'menu': [{'nm': 'Bintang Bremer', 'cnt': '1',...","{'menu': [{'nm': 'Bintang Bremer', 'cnt': '1',..."
4,5,"{'menu': {'nm': 'BASO BIHUN', 'unitprice': '43...","{'menu': [{'nm': 'BASO BIHUN', 'cnt': '1', 'pr..."
...,...,...,...
994,996,"{'menu': [{'nm': 'BASO TAHU', 'unitprice': '43...","{'menu': [{'nm': 'BASO TAHU', 'cnt': '1', 'pri..."
995,997,"{'menu': {'nm': 'BBQ Chicken', 'cnt': '1', 'pr...","{'menu': [{'nm': 'BBQ Chicken', 'cnt': '1', 'p..."
996,998,"{'menu': [{'nm': 'BUBUR GO', 'unitprice': '20....","{'menu': [{'nm': 'BUBUR GO', 'cnt': '2', 'pric..."
997,999,"{'menu': [{'nm': 'BLACK PAPPER MEATBALL PAS', ...","{'menu': [{'nm': 'BLACK PEPPER MEATBALL PAS', ..."


In [31]:
def flatten_json(y, prefix=""):
    """Flatten nested JSON into dot notation keys."""
    out = {}

    def flatten(x, name=""):
        if isinstance(x, dict):
            for a in x:
                flatten(x[a], f"{name}{a}.")
        elif isinstance(x, list):
            for i, a in enumerate(x):
                flatten(a, f"{name}{i}.")
        else:
            out[name[:-1]] = x

    flatten(y, prefix)
    return out


def fix_data_type_mismatch(gt, pred):
    if "menu" in gt and "menu" in pred:
        if isinstance(gt["menu"], dict) and isinstance(pred["menu"], list):
            gt["menu"] = [gt["menu"]]

        if isinstance(gt["menu"], list) and isinstance(pred["menu"], dict):
            pred["menu"] = [pred["menu"]]

    return gt, pred


def calculate_invoice_accuracies(ground_truth_list, response_list):
    """Calculate per-invoice accuracy and return a DataFrame."""
    invoice_metrics = []
    for i in range(len(ground_truth_list)):
        gt = ground_truth_list[i]
        pred = response_list[i]
        # If response is an OpenAI object, parse output_text
        if hasattr(pred, "output_text"):
            pred = json.loads(pred.output_text)

        gt, pred = fix_data_type_mismatch(gt, pred)
        gt_flat = flatten_json(gt)
        # gt_flat = apply_postprocessing(gt_flat)
        pred_flat = flatten_json(pred)
        # pred_flat = apply_postprocessing(pred_flat)
        total_keys = len(gt_flat)
        matched_keys = sum(
            str(gt_flat[k]).strip() == str(pred_flat.get(k, "")).strip()
            for k in gt_flat
        )
        accuracy = matched_keys / total_keys if total_keys > 0 else 0.0
        invoice_metrics.append(
            {
                "invoice_no": i,
                "total_keys": total_keys,
                "matched_keys": matched_keys,
                "accuracy": accuracy,
            }
        )
    invoice_metrics_df = pd.DataFrame(invoice_metrics)
    return invoice_metrics_df


In [32]:
invoice_result_df = calculate_invoice_accuracies(
    final_result_df["gt_parse"].tolist(), final_result_df["response"].tolist()
)

In [33]:
invoice_result_df

,invoice_no,total_keys,matched_keys,accuracy
0,0,71,70,0.985915
1,1,24,21,0.875000
2,2,21,21,1.000000
3,3,14,14,1.000000
4,4,9,8,0.888889
...,...,...,...,...
994,994,13,9,0.692308
995,995,11,10,0.909091
996,996,13,4,0.307692
997,997,13,12,0.923077
